## Text input

https://platform.openai.com/docs/models

In [12]:
from dotenv import load_dotenv

load_dotenv()

True

In [13]:
from langchain.agents import create_agent

agent = create_agent(
    model='gpt-5-nano',
    system_prompt="You are a helpful assistant that describes images accurately. If something is unclear, say so instead of guessing.",
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

## Image input

In [2]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.jpg', multiple=False)
display(uploader)

FileUpload(value=(), accept='.jpg', description='Upload')

In [3]:
print(uploader.value)

({'name': 'draught-beer-png-mug.jpg', 'type': 'image/jpeg', 'size': 12416460, 'content': <memory at 0x1059d9d80>, 'last_modified': datetime.datetime(2026, 11, 16, 20, 2, 15, 727000, tzinfo=datetime.timezone.utc)},)


In [6]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

print(uploaded_file)

# This is a memoryview
content_mv = uploaded_file["content"]

print(content_mv)

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

print(img_bytes)

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")
print(img_b64)

{'name': 'draught-beer-png-mug.jpg', 'type': 'image/jpeg', 'size': 12416460, 'content': <memory at 0x1059d9d80>, 'last_modified': datetime.datetime(2026, 11, 16, 20, 2, 15, 727000, tzinfo=datetime.timezone.utc)}
b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x01,\x01,\x00\x00\xff\xe1\x01\xceExif\x00\x00MM\x00*\x00\x00\x00h\x00\x05\x01\x1a\x00\x05\x00\x00\x00\x01\x00\x00\x00J\x01\x1b\x00\x05\x00\x00\x00\x01\x00\x00\x00R\x01(\x00\x03\x00\x00\x00\x01\x00\x02\x00\x00\x02\x13\x00\x03\x00\x00\x00\x01\x00\x01\x00\x00\x82\x98\x00\x02\x00\x00\x00\x0e\x00\x00\x00Z\x00\x00\x00\x00\x00\x00\x01,\x00\x00\x00\x01\x00\x00\x01,\x00\x00\x00\x01Rawpixel Ltd.\x00\x00\x0b\x01\x0e\x00\x02\x00\x00\x00!\x00\x00\x00\xf2\x01\x12\x00\x03\x00\x00\x00\x01\x00\x01\x00\x00\x01\x1a\x00\x05\x00\x00\x00\x01\x00\x00\x00J\x01\x1b\x00\x05\x00\x00\x00\x01\x00\x00\x00R\x01(\x00\x03\x00\x00\x00\x01\x00\x02\x00\x00\x011\x00\x02\x00\x00\x00\t\x00\x00\x01\x14\x012\x00\x02\x00\x00\x00\x14\x00\x00\x01\x1e\x01;\x00\x02\x00\x00\x00

In [15]:
from langchain.messages import HumanMessage


multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this picture"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

This is a close-up photo of a beer in a glass mug. The beer is a bright golden‑amber color with a thick, foamy white head filled with bubbles. The mug and glass are covered in condensation, emphasizing how cold it is. The handle is visible on the left, and the background is clean and white, giving a crisp, high‑key look. Overall, it conveys a refreshing, inviting vibe.


## Audio input

In [16]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

ModuleNotFoundError: No module named 'tqdm'

In [ ]:
agent = create_agent(
    model='gpt-audio',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)